# Actividad 5: clasificación de sentimiento

Este notebook presenta la calidad de datos, la preparación ya registrada y las evaluaciones de Regresión Logística y Random Forest. La lógica de producción está en `fuentes/datos_prep.py`, `fuentes/train.py`, `fuentes/auditoria_fase3.py` y `fuentes/mlflow_tracking.py`; este notebook no duplica sus pipelines.

> La celda de reproducción vuelve a ejecutar las búsquedas completas en carpetas temporales. Compara con los resultados archivados y no sobrescribe las evidencias validadas. Selecciona el kernel `.venv` antes de ejecutar.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'fuentes').is_dir() and (p / 'README.md').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abre este notebook desde el proyecto Actividad5 para localizar la raíz.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Raíz del proyecto:', ROOT)

## Preparación y procedencia

Descarga `Tweets.csv` desde la fuente Kaggle citada en el README y colócalo en `datos/datos_ini/Tweets.csv`. Si aún no existe la tabla limpia, ejecuta desde la raíz ` .\.venv\Scripts\python.exe fuentes\datos_prep.py`. El script registra reglas, calidad y SHA-256; este notebook lee esos artefactos y no regenera archivos históricos.

In [ ]:
import hashlib
import json
import pandas as pd

clean_path = ROOT / 'datos' / 'datos_limp' / 'Tweets_limpios.csv'
manifest_path = ROOT / 'datos' / 'datos_limp' / 'manifiesto.json'
quality_path = ROOT / 'datos' / 'datos_limp' / 'calidad_datos.json'
if not clean_path.is_file() or not manifest_path.is_file() or not quality_path.is_file():
    raise FileNotFoundError('Falta la versión limpia o sus reportes. Descarga el CSV y ejecuta fuentes/datos_prep.py según el README.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
actual_hash = hashlib.sha256(clean_path.read_bytes()).hexdigest()
if actual_hash != manifest['processed']['sha256']:
    raise ValueError('El hash de la tabla limpia no coincide con el manifiesto; detengo la reproducción.')
quality = json.loads(quality_path.read_text(encoding='utf-8'))
data = pd.read_csv(clean_path)
print('Filas/columnas:', data.shape)
print('SHA-256 limpio:', actual_hash)
print('Nulos:', int(data.isna().sum().sum()))
print('Duplicados texto-etiqueta:', int(data.duplicated(['text', 'airline_sentiment']).sum()))

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

print('Columnas usadas:', list(data.columns))
print('Reglas:', *manifest['cleaning_rules'], sep='\n- ')
print('Conteos de clase tras limpieza:')
display(data['airline_sentiment'].value_counts().rename_axis('clase').to_frame('registros'))
data['airline_sentiment'].value_counts().reindex(['negative', 'neutral', 'positive']).plot.bar(color=['#c44e52', '#8172b2', '#55a868'])
plt.title('Distribución de clases en los datos limpios')
plt.ylabel('Registros')
plt.tight_layout()
plt.show()

## Entrenamiento y evaluación

`train.py` separa los datos con estratificación 80/20 (semilla 42), ajusta TF-IDF dentro del pipeline y usa GridSearchCV con cinco folds para escoger hiperparámetros por F1 macro. El test se reserva para evaluación. La evaluación agrupada emplea StratifiedGroupKFold para impedir que un texto exacto aparezca en ambos subconjuntos; es complementaria y no se mezcla con el escenario original. No se calculó significancia estadística.

In [ ]:
original_path = ROOT / 'reportes' / 'entrenamiento' / 'metricas_experimentos.json'
grouped_path = ROOT / 'reportes' / 'auditoria_fase3' / 'evaluacion_agrupada' / 'evaluacion_agrupada.json'
if not original_path.is_file() or not grouped_path.is_file():
    raise FileNotFoundError('Faltan los resultados validados; ejecuta primero entrenamiento y auditoría, siguiendo el README.')
original = json.loads(original_path.read_text(encoding='utf-8'))
grouped = json.loads(grouped_path.read_text(encoding='utf-8'))
rows = []
for scenario, report in [('evaluacion_original', original), ('evaluacion_agrupada', grouped)]:
    for model_name, result in report['models'].items():
        metric = result['test_metrics']
        cv_mean = result.get('cv_selected_candidate', {}).get('mean_f1_macro', result.get('cv_mean_f1_macro'))
        cv_std = result.get('cv_selected_candidate', {}).get('std_f1_macro', result.get('cv_std_f1_macro_sample'))
        rows.append({'escenario': scenario, 'modelo': model_name, 'accuracy': metric['accuracy'], 'precision_macro': metric['precision_macro'], 'recall_macro': metric['recall_macro'], 'f1_macro': metric['f1_macro'], 'f1_weighted': metric['f1_weighted'], 'roc_auc_ovr_macro': metric['roc_auc_ovr_macro'], 'cv_f1_macro_mean': cv_mean, 'cv_f1_macro_std': cv_std})
results_table = pd.DataFrame(rows)
display(results_table)

In [ ]:
# Reproducción opcional de las cuatro búsquedas. Es costosa y escribe solo en un directorio temporal.
import math
import tempfile
from fuentes.train import run_experiments
from fuentes.auditoria_fase3 import grouped_evaluation
from fuentes.datos_prep import load_dataset

source_rows, _ = load_dataset(clean_path)
with tempfile.TemporaryDirectory(prefix='actividad5_notebook_') as temp:
    temp = Path(temp)
    reproduced_original = run_experiments(clean_path, temp / 'original', temp / 'original_evidence')
    reproduced_grouped = grouped_evaluation(source_rows, temp / 'grouped', temp / 'grouped_evidence')
    assert reproduced_grouped['protocol']['train_test_text_overlap'] == 0
    for scenario, reproduced, archived in [('evaluacion_original', reproduced_original, original), ('evaluacion_agrupada', reproduced_grouped, grouped)]:
        for name, result in reproduced['models'].items():
            expected = archived['models'][name]
            assert json.loads(json.dumps(result['best_params'])) == expected['best_params'], (scenario, name, 'params')
            for metric, value in result['test_metrics'].items():
                if isinstance(value, (int, float)):
                    assert math.isclose(value, expected['test_metrics'][metric], rel_tol=1e-12, abs_tol=1e-12), (scenario, name, metric)
print('Reproducción coincidente con los resultados archivados; artefactos temporales eliminados al cerrar la celda.')

## Seguimiento con MLflow

La integración registra cuatro runs en `Actividad5_Clasificacion_Sentimiento`, identificados por escenario y modelo. Para abrir la interfaz en una terminal: ` .\.venv\Scripts\mlflow.exe server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000` y visita <http://127.0.0.1:5000>. Los archivos locales de base de datos y artefactos no se suben a Git.

In [ ]:
tracking_db = ROOT / 'mlflow.db'
if not tracking_db.is_file():
    print('No hay mlflow.db local. Después de preparar datos, entrenar y auditar, ejecuta fuentes/mlflow_tracking.py para registrar las cuatro ejecuciones.')
else:
    import mlflow
    from mlflow.tracking import MlflowClient
    mlflow.set_tracking_uri(tracking_db.resolve().as_uri().replace('file:///', 'sqlite:///'))
    client = MlflowClient()
    experiment = client.get_experiment_by_name('Actividad5_Clasificacion_Sentimiento')
    if experiment is None:
        print('No se encontró el experimento esperado en esta base local.')
    else:
        runs = client.search_runs([experiment.experiment_id], max_results=100)
        run_table = pd.DataFrame([{'run_id': run.info.run_id, 'escenario': run.data.tags.get('evaluation_type'), 'modelo': run.data.params.get('model'), 'f1_macro': run.data.metrics.get('f1_macro'), 'accuracy': run.data.metrics.get('accuracy')} for run in runs])
        display(run_table.sort_values(['escenario', 'modelo']))

## Limitaciones y cierre

Los datos están desbalanceados; existen textos repetidos y 25 grupos con etiquetas conflictivas. El split original comparte 71 textos entre train y test, por lo que se conserva también la evaluación agrupada, que no sustituye a la original. No se hizo una prueba de significancia, no se autentica la descarga local con un checksum editorial y el rendimiento no se ha validado en producción. Las capturas reales de MLflow, cuando estén disponibles en `evidencias/mlflow/`, complementan la tabla de runs; este notebook no genera capturas simuladas.